# Retail Product Recognition and Checkout Price Verification

**A reproducible object-detection workflow for the RPC dataset**

This notebook prepares the official train/validation/test splits, audits COCO annotations, converts them to Ultralytics YOLO format, fine-tunes a pretrained detector, evaluates both localization and checkout counts, and offers a price-catalog integration point.

> **Important:** this is an academic/demo workflow. The notebook does not invent prices or treat a coarse product family as a SKU. The checkout amount stays incomplete until a verified store catalog maps every detected product class to a real item and price.

### Learning goals

1. Understand the difference between exemplar product photos and cluttered checkout images.
2. Preserve official splits and inspect class coverage before training.
3. Validate bounding boxes and labels before a long GPU run.
4. Report object-detection metrics and exact per-image item-count metrics.
5. Keep SKU identity, confidence, and price resolution explicit in a checkout pipeline.

### Runtime

Use a Kaggle Notebook with a GPU accelerator enabled. The default profile trains on the full official train split and validates on the full official validation split. A separately labeled quick profile is available for checking the pipeline; its results are exploratory and are not the final model. The 24,000-image official test split is prepared and evaluated only in the final optional section.

### Dataset and license

The [RPC project page](https://rpc-dataset.github.io/) describes 53,739 single-product exemplar training images, 6,000 multi-product validation images, and 24,000 multi-product test images. It lists the dataset license as **CC BY-NC-SA 4.0** and says other use requires contacting the authors. Do not commit dataset images or annotations to GitHub. Review the license before redistribution or any commercial use.

### References informing the workflow

- The [RPC dataset paper/project](https://rpc-dataset.github.io/) highlights the domain gap between controlled exemplar images and cluttered checkout scenes.
- A 2024 checkout study fine-tunes YOLO models and reports gains from paired-image augmentation and a modified NMS strategy: [Solving automatic check-out with fine-tuned YOLO models](https://doi.org/10.1016/j.procs.2024.09.644). This notebook starts with a reproducible baseline; those additions belong in a measured ablation, not an assumed improvement.
- A 2024 ECCV paper shows detector confidence calibration needs to be evaluated jointly with accuracy: [On Calibration of Object Detectors](https://arxiv.org/abs/2405.20459). Here, the decision confidence is selected on validation checkout images and frozen before test evaluation.
- A 2025 CVPR workshop paper explores retrieval-augmented vision-language methods for fine-grained products: [A Visual RAG Pipeline for Few-Shot Fine-Grained Product Classification](https://openaccess.thecvf.com/content/CVPR2025W/FGVC/html/Lamm_A_Visual_RAG_Pipeline_for_Few-Shot_Fine-Grained_Product_Classification_CVPRW_2025_paper.html). A catalog retrieval or OCR branch is a promising follow-up when SKUs change frequently.
- The current [Ultralytics detection guide](https://docs.ultralytics.com/tasks/detect/) documents pretrained model training, prediction, and validation.

---

## 1. Environment and reproducibility

Kaggle commonly provides PyTorch and data-science libraries already. The next cell installs Ultralytics and KaggleHub only when they are missing, then imports the libraries used below. Keep the notebook's Python package versions in the final README when publishing so another user can reproduce the run.

In [ ]:
import importlib.util

missing = [package for package in ("ultralytics", "kagglehub") if importlib.util.find_spec(package) is None]
if missing:
    %pip install -q ultralytics kagglehub

### Reproducibility controls

Set one seed, choose a clearly labeled run profile, and keep the official train/validation/test splits separate. The default `full` profile uses the complete 53,739-image training split and all 6,000 validation images. Choose `quick` only to smoke-check the pipeline on a smaller subset before committing to the full training run.

The small pretrained **YOLO26n** checkpoint is selected as a fast, current Ultralytics baseline. If your Kaggle environment does not yet include that model generation, upgrade Ultralytics or change `MODEL_WEIGHTS` to a checkpoint supported by your installed version. Training automatically uses a conservative batch size based on available GPU memory.

In [ ]:
from collections import Counter, defaultdict
import os
from pathlib import Path
import json
import random
import shutil
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageDraw
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATASET_HANDLE = "diyer22/retail-product-checkout-dataset"
MODEL_WEIGHTS = "yolo26n.pt"
TRAINING_PROFILE = "full"  # Use "quick" only for a short exploratory run.
if TRAINING_PROFILE == "full":
    MAX_TRAIN_IMAGES = None  # All official training images (about 53,739).
    MAX_VAL_IMAGES = None  # All official validation images (about 6,000).
    EPOCHS = 50
elif TRAINING_PROFILE == "quick":
    MAX_TRAIN_IMAGES = 2_000
    MAX_VAL_IMAGES = 1_000
    EPOCHS = 20
else:
    raise ValueError("TRAINING_PROFILE must be 'full' or 'quick'.")
PREPARE_OFFICIAL_TEST = False  # Turn on only after model selection is complete.
IMAGE_SIZE = 640
PREDICTION_CONFIDENCE_FLOOR = 0.001
NMS_IOU = 0.70
OUTPUT_ROOT = Path("/kaggle/working/retail_price_verification")
YOLO_DATASET_ROOT = OUTPUT_ROOT / "yolo_dataset"
RUN_PROJECT = OUTPUT_ROOT / "runs"

print(f"Python device: {'CUDA: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Profile: {TRAINING_PROFILE} | train/validation images: {MAX_TRAIN_IMAGES or 'all'} / {MAX_VAL_IMAGES or 'all'} | image size: {IMAGE_SIZE} | epochs: {EPOCHS}")

## 2. Find the dataset without assuming one machine's path

On Kaggle, attach **Retail Product Checkout Dataset** using the notebook's **Add Input** control. The notebook first checks known Kaggle mount paths and avoids recursively searching the whole input volume when possible. If the data is not mounted (for example, in another Jupyter environment), it falls back to KaggleHub's dataset download. If Kaggle exposes duplicate extracted/archive copies, the path resolver checks a small sample of referenced files and chooses the closest complete pair. The test split is only discovered and loaded when its optional evaluation is enabled.

In [ ]:
import kagglehub

KAGGLE_INPUT_ROOTS = [
    Path("/kaggle/input/datasets/diyer22/retail-product-checkout-dataset"),
    Path("/kaggle/input/retail-product-checkout-dataset"),
]
DATASET_ROOT = next((root for root in KAGGLE_INPUT_ROOTS if root.is_dir()), None)
if DATASET_ROOT is None and Path("/kaggle/input").exists():
    # Fallback for a renamed Kaggle slug; this only runs when the known mounts are absent.
    for candidate in Path("/kaggle/input").iterdir():
        if candidate.is_dir() and any(candidate.rglob("instances_train2019.json")):
            DATASET_ROOT = candidate
            break
if DATASET_ROOT is None:
    DATASET_ROOT = Path(kagglehub.dataset_download(DATASET_HANDLE))

print("Dataset root:", DATASET_ROOT.resolve())
print("Top-level entries:")
for entry in sorted(DATASET_ROOT.iterdir())[:20]:
    print(" •", entry.name, "(directory)" if entry.is_dir() else f"({entry.stat().st_size:,} bytes)")

SPLITS = ("train", "val", "test") if PREPARE_OFFICIAL_TEST else ("train", "val")
SOURCE = {}
for split in SPLITS:
    annotation_candidates = list(DATASET_ROOT.rglob(f"instances_{split}2019.json"))
    image_candidates = [p for p in DATASET_ROOT.rglob(f"{split}2019") if p.is_dir()]
    if not annotation_candidates or not image_candidates:
        raise FileNotFoundError(
            f"Could not locate {split}2019. Found {len(annotation_candidates)} annotation "
            f"files and {len(image_candidates)} image directories under {DATASET_ROOT}. "
            "Check that the complete RPC dataset was attached."
        )

    # Kaggle may expose both an extracted copy and a nested/archive copy. Check
    # a small deterministic sample instead of recursively indexing every image
    # in a potentially huge mounted dataset.
    scored_pairs = []
    for annotation_path in annotation_candidates:
        annotation_data = json.loads(annotation_path.read_text(encoding="utf-8"))
        referenced_names = [image["file_name"] for image in annotation_data["images"][:128]]
        if not referenced_names:
            continue
        for image_dir in image_candidates:
            found = sum(
                (image_dir / file_name).is_file()
                or (image_dir / Path(file_name).name).is_file()
                for file_name in referenced_names
            )
            coverage = found / len(referenced_names)
            common_parts = Path(os.path.commonpath([str(annotation_path.parent), str(image_dir)] )).parts
            distance = len(annotation_path.parent.parts) + len(image_dir.parts) - 2 * len(common_parts)
            scored_pairs.append((coverage, -distance, len(common_parts), annotation_path, image_dir))

    if not scored_pairs:
        raise FileNotFoundError(f"No usable annotation/image pair found for {split} under {DATASET_ROOT}.")
    best_score = max(pair[:3] for pair in scored_pairs)
    best_pairs = [pair for pair in scored_pairs if pair[:3] == best_score]
    if best_score[0] < 0.95:
        candidates = "\n".join(f"  {row[3]} <-> {row[4]} (coverage={row[0]:.1%})" for row in sorted(scored_pairs, key=lambda row: row[:3], reverse=True)[:5])
        raise FileNotFoundError(
            f"No complete {split} image directory matched the COCO annotations. Best candidates:\n{candidates}"
        )
    selected = sorted(best_pairs, key=lambda row: (str(row[3]), str(row[4])))[0]
    SOURCE[split] = {"json": selected[3], "images": selected[4]}
    if len(annotation_candidates) > 1 or len(image_candidates) > 1:
        print(f"Resolved {split} duplicate candidates using referenced-file coverage and path proximity.")

for split, paths in SOURCE.items():
    print(f"{split:5s} | annotations: {paths['json']} | images: {paths['images']}")

## 3. Inspect the dataset and preserve its official split design

RPC uses a useful but challenging protocol: training images show one product in a controlled setting; validation and test images show multiple products on a checkout surface. A random split of training images would leak the easy single-product setup into evaluation and would not answer whether the model works at checkout. We keep the official splits intact.

The train subset cap below is sampled by the image's dominant class to retain class coverage. Validation remains complete, and the final test split is untouched until model selection and confidence-threshold tuning are finished.

In [ ]:
def read_coco(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        data = json.load(handle)
    required = {"images", "annotations", "categories"}
    missing_keys = required.difference(data)
    if missing_keys:
        raise ValueError(f"{path.name} is missing COCO keys: {sorted(missing_keys)}")
    return data


COCO = {split: read_coco(paths["json"]) for split, paths in SOURCE.items()}
category_rows = sorted(COCO["train"]["categories"], key=lambda row: row["id"])
CLASS_NAMES = [str(row["name"]) for row in category_rows]
COCO_ID_TO_CLASS = {int(row["id"]): index for index, row in enumerate(category_rows)}
CLASS_TO_COCO_ID = {index: category_id for category_id, index in COCO_ID_TO_CLASS.items()}

for split in SPLITS:
    current_categories = sorted(COCO[split]["categories"], key=lambda row: row["id"])
    current_signature = [(int(row["id"]), str(row["name"])) for row in current_categories]
    train_signature = [(int(row["id"]), str(row["name"])) for row in category_rows]
    if current_signature != train_signature:
        raise ValueError(f"Category IDs or names differ in the official {split} split.")

summary = []
for split in SPLITS:
    data = COCO[split]
    summary.append({
        "split": split,
        "images": len(data["images"]),
        "annotations": len(data["annotations"]),
        "classes": len(data["categories"]),
        "objects per image": round(len(data["annotations"]) / max(len(data["images"]), 1), 2),
    })
display(pd.DataFrame(summary).set_index("split"))
print(f"Class names: {len(CLASS_NAMES)}; first 12: {CLASS_NAMES[:12]}")

### Annotation integrity checks

Before copying gigabytes of files, check for duplicate image IDs, annotations that point to missing images/categories, invalid dimensions, and malformed or out-of-frame boxes. This makes data defects visible before a training run. Coordinates are clipped to image bounds during conversion; degenerate boxes and crowd annotations are excluded and counted.

In [ ]:
def audit_coco(data: dict, split: str) -> dict:
    image_ids = [int(row["id"]) for row in data["images"]]
    image_id_set = set(image_ids)
    image_by_id = {int(row["id"]): row for row in data["images"]}
    category_ids = {int(row["id"]) for row in data["categories"]}
    duplicates = len(image_ids) - len(image_id_set)
    invalid_dimensions = sum(int(row.get("width", 0)) <= 0 or int(row.get("height", 0)) <= 0 for row in data["images"])
    unknown_images = sum(int(row.get("image_id", -1)) not in image_id_set for row in data["annotations"])
    unknown_categories = sum(int(row.get("category_id", -1)) not in category_ids for row in data["annotations"])
    malformed_boxes = 0
    out_of_frame = 0
    for ann in data["annotations"]:
        bbox = ann.get("bbox")
        if not isinstance(bbox, list) or len(bbox) != 4 or not np.isfinite(bbox).all():
            malformed_boxes += 1
            continue
        x, y, width, height = map(float, bbox)
        if width <= 0 or height <= 0:
            malformed_boxes += 1
        image = image_by_id.get(int(ann.get("image_id", -1)))
        if image is not None and (x < 0 or y < 0 or x + width > image["width"] or y + height > image["height"]):
            out_of_frame += 1
    return {
        "split": split, "duplicate image IDs": duplicates, "bad image dimensions": invalid_dimensions,
        "annotations with unknown images": unknown_images, "annotations with unknown classes": unknown_categories,
        "malformed/degenerate boxes": malformed_boxes, "boxes outside frame (will be clipped)": out_of_frame,
    }


audit = pd.DataFrame([audit_coco(COCO[split], split) for split in SPLITS]).set_index("split")
display(audit)
assert (audit["duplicate image IDs"] == 0).all(), "Image IDs must be unique within each split."
assert (audit["annotations with unknown images"] == 0).all(), "Some annotations point to missing images."
assert (audit["annotations with unknown classes"] == 0).all(), "Some annotations point to missing categories."
assert (audit["bad image dimensions"] == 0).all(), "Some images have invalid dimensions."

## 4. Inspect class imbalance and raw bounding boxes

The RPC task is fine-grained: many classes are visually similar, while the product counts per class can be uneven. We plot the annotation distribution and inspect labeled images. This is a diagnostic, not a promise that a detector can distinguish every package variant; rare classes and confusable pairs need their own error review.

In [ ]:
train_class_counts = Counter(
    CLASS_NAMES[COCO_ID_TO_CLASS[int(ann["category_id"])]]
    for ann in COCO["train"]["annotations"]
    if int(ann.get("category_id", -1)) in COCO_ID_TO_CLASS
)
class_count_frame = pd.DataFrame({"class": CLASS_NAMES, "train instances": [train_class_counts[name] for name in CLASS_NAMES]})
class_count_frame = class_count_frame.sort_values("train instances", ascending=False).reset_index(drop=True)
top_classes = class_count_frame.head(30).sort_values("train instances", ascending=True)

plt.figure(figsize=(12, 11))
plt.barh(top_classes["class"], top_classes["train instances"], color="#287C8E")
plt.title("RPC training annotations: 30 most frequent product classes", fontsize=15, pad=14)
plt.xlabel("Number of instances", fontsize=12)
plt.ylabel("Product class", fontsize=12)
plt.grid(axis="x", alpha=0.22)
plt.tight_layout()
plt.show()
display(class_count_frame.set_index("class").rename_axis("All product classes").style.format({"train instances": "{:,}"}))

annotation_index = defaultdict(list)
for ann in COCO["train"]["annotations"]:
    annotation_index[int(ann["image_id"])].append(ann)

preview_records = [row for row in COCO["train"]["images"] if annotation_index.get(int(row["id"]))]
random.Random(SEED).shuffle(preview_records)
preview_records = preview_records[:4]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, image_record in zip(axes.flat, preview_records):
    image_path = SOURCE["train"]["images"] / image_record["file_name"]
    with Image.open(image_path).convert("RGB") as image:
        draw = ImageDraw.Draw(image)
        for ann in annotation_index[int(image_record["id"])]:
            x, y, width, height = map(float, ann["bbox"])
            draw.rectangle((x, y, x + width, y + height), outline="#E63946", width=3)
            draw.text((x + 3, y + 3), CLASS_NAMES[COCO_ID_TO_CLASS[int(ann["category_id"])]], fill="#E63946")
        ax.imshow(image)
    ax.set_title(f"image_id={image_record['id']}", fontsize=11)
    ax.axis("off")
for ax in axes.flat[len(preview_records):]:
    ax.axis("off")
plt.suptitle("Raw training examples with COCO boxes", fontsize=16)
plt.tight_layout()
plt.show()

## 5. Convert COCO to YOLO with the original split boundaries

YOLO detection labels store one object per line as **class index, normalized x-center, y-center, width, height**. The conversion uses a single class-index map across all splits, preserves image IDs in output filenames, clips boxes to image dimensions, and writes a portable dataset YAML.

The default `full` profile uses every official training image and every official checkout validation image. Set `TRAINING_PROFILE = "quick"` only for a faster exploratory run (2,000 train and 1,000 validation images). Quick-profile metrics are not full-split benchmark results. Either profile keeps the official splits intact; we never create validation images from training exemplars.

In [ ]:
def select_training_images(images: list, annotations: list, max_images: int | None, seed: int) -> list:
    if max_images is None or max_images >= len(images):
        return list(images)
    if max_images <= 0:
        raise ValueError("max_images must be positive or None")

    labels_by_image = defaultdict(list)
    for ann in annotations:
        if not ann.get("iscrowd", 0):
            labels_by_image[int(ann["image_id"])].append(int(ann["category_id"]))
    class_frequency = Counter(label for labels in labels_by_image.values() for label in set(labels))
    eligible = [row for row in images if labels_by_image.get(int(row["id"]))]
    strata = []
    for row in eligible:
        labels = set(labels_by_image[int(row["id"])])
        # For the RPC exemplar set this is the only class; for multi-object images, use the rarest present class.
        strata.append(min(labels, key=lambda label: class_frequency[label]))

    try:
        from sklearn.model_selection import train_test_split
        chosen, _ = train_test_split(
            eligible, train_size=min(max_images, len(eligible)), random_state=seed, stratify=strata
        )
    except (ValueError, ImportError) as error:
        warnings.warn(f"Could not stratify the requested sample ({error}); using seeded random sampling.")
        chosen = random.Random(seed).sample(eligible, min(max_images, len(eligible)))
    return sorted(chosen, key=lambda row: int(row["id"]))


def convert_coco_split(split: str, max_images: int | None = None) -> dict:
    source = SOURCE[split]
    data = COCO[split]
    split_cap = {"train": MAX_TRAIN_IMAGES, "val": MAX_VAL_IMAGES}.get(split)
    selected_images = select_training_images(data["images"], data["annotations"], split_cap, SEED) if split in ("train", "val") else list(data["images"])
    image_ids = {int(row["id"]) for row in selected_images}
    ann_by_image = defaultdict(list)
    for ann in data["annotations"]:
        image_id = int(ann["image_id"])
        if image_id in image_ids:
            ann_by_image[image_id].append(ann)

    image_out = YOLO_DATASET_ROOT / "images" / split
    label_out = YOLO_DATASET_ROOT / "labels" / split
    image_out.mkdir(parents=True, exist_ok=True)
    label_out.mkdir(parents=True, exist_ok=True)
    output_name_to_id = {}
    diagnostics = Counter()

    for record in selected_images:
        image_id = int(record["id"])
        source_image = source["images"] / record["file_name"]
        if not source_image.is_file():
            # Some COCO manifests use only the basename while the archive contains nested folders.
            matches = list(source["images"].rglob(Path(record["file_name"]).name))
            if len(matches) != 1:
                diagnostics["missing or ambiguous source images"] += 1
                continue
            source_image = matches[0]

        safe_name = f"{image_id}__{Path(record['file_name']).name}"
        destination_image = image_out / safe_name
        shutil.copy2(source_image, destination_image)
        output_name_to_id[safe_name] = image_id
        label_lines = []
        image_width, image_height = int(record["width"]), int(record["height"])
        for ann in ann_by_image.get(image_id, []):
            if ann.get("iscrowd", 0):
                diagnostics["crowd annotations skipped"] += 1
                continue
            category_id = int(ann["category_id"])
            if category_id not in COCO_ID_TO_CLASS:
                diagnostics["unknown category annotations skipped"] += 1
                continue
            bbox = ann.get("bbox", [])
            if len(bbox) != 4 or not np.isfinite(bbox).all():
                diagnostics["malformed boxes skipped"] += 1
                continue
            x, y, width, height = map(float, bbox)
            x1, y1 = max(0.0, x), max(0.0, y)
            x2, y2 = min(float(image_width), x + width), min(float(image_height), y + height)
            clipped_width, clipped_height = x2 - x1, y2 - y1
            if clipped_width <= 0 or clipped_height <= 0:
                diagnostics["degenerate boxes skipped"] += 1
                continue
            if x1 != x or y1 != y or x2 != x + width or y2 != y + height:
                diagnostics["boxes clipped to frame"] += 1
            center_x = ((x1 + x2) / 2) / image_width
            center_y = ((y1 + y2) / 2) / image_height
            normalized_width = clipped_width / image_width
            normalized_height = clipped_height / image_height
            class_index = COCO_ID_TO_CLASS[category_id]
            label_lines.append(
                f"{class_index} {center_x:.6f} {center_y:.6f} {normalized_width:.6f} {normalized_height:.6f}"
            )

        label_path = label_out / f"{Path(safe_name).stem}.txt"
        label_path.write_text("\n".join(label_lines), encoding="utf-8")
        diagnostics["images converted"] += 1
        diagnostics["objects converted"] += len(label_lines)

    if diagnostics["missing or ambiguous source images"]:
        raise FileNotFoundError(f"{diagnostics['missing or ambiguous source images']} source images could not be found.")
    return {"split": split, "selected_images": len(selected_images), "output_name_to_id": output_name_to_id, "diagnostics": diagnostics}


train_conversion = convert_coco_split("train")
val_conversion = convert_coco_split("val")
CONVERSION = {"train": train_conversion, "val": val_conversion}

dataset_yaml = {
    "path": str(YOLO_DATASET_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": CLASS_NAMES,
}
DATA_YAML_PATH = YOLO_DATASET_ROOT / "data.yaml"
DATA_YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
DATA_YAML_PATH.write_text(yaml.safe_dump(dataset_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")

for split, conversion in CONVERSION.items():
    print(f"{split}: {dict(conversion['diagnostics'])}")
print("YOLO dataset config:", DATA_YAML_PATH)

### Conversion audit

Check that the generated image/label pairs exist, normalized values fall inside the allowed range, and the class IDs are contiguous from zero to the number of classes minus one. This catches common conversion bugs before GPU time is spent.

In [ ]:
def audit_yolo_split(split: str) -> dict:
    images_dir = YOLO_DATASET_ROOT / "images" / split
    labels_dir = YOLO_DATASET_ROOT / "labels" / split
    image_files = [p for p in images_dir.rglob("*") if p.is_file()]
    invalid_rows = 0
    invalid_class_ids = 0
    missing_labels = 0
    objects = 0
    for image_path in image_files:
        label_path = labels_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            missing_labels += 1
            continue
        for line in label_path.read_text(encoding="utf-8").splitlines():
            fields = line.split()
            if len(fields) != 5:
                invalid_rows += 1
                continue
            class_id = int(fields[0])
            values = np.asarray(fields[1:], dtype=float)
            objects += 1
            invalid_class_ids += not (0 <= class_id < len(CLASS_NAMES))
            invalid_rows += not (np.isfinite(values).all() and (values >= 0).all() and (values <= 1).all())
            invalid_rows += not (values[2] > 0 and values[3] > 0)
    return {"split": split, "images": len(image_files), "objects": objects, "empty/missing labels": missing_labels,
            "invalid rows": invalid_rows, "invalid class IDs": invalid_class_ids}


converted_audit = pd.DataFrame([audit_yolo_split(split) for split in ("train", "val")]).set_index("split")
display(converted_audit)
assert (converted_audit["invalid rows"] == 0).all()
assert (converted_audit["invalid class IDs"] == 0).all()
assert (converted_audit["empty/missing labels"] == 0).all(), "RPC train/val are expected to have product labels."

### Visual check of converted labels

The conversion audit checks numeric validity; a visual sample checks that the class index and box still align with the product. Always inspect more than one image, especially after changing category ordering or image paths.

In [ ]:
def show_yolo_samples(split: str = "val", number: int = 4) -> None:
    images_dir = YOLO_DATASET_ROOT / "images" / split
    image_files = sorted(p for p in images_dir.rglob("*") if p.is_file())
    sample = random.Random(SEED).sample(image_files, min(number, len(image_files)))
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    for ax, image_path in zip(axes.flat, sample):
        with Image.open(image_path).convert("RGB") as image:
            draw = ImageDraw.Draw(image)
            label_path = YOLO_DATASET_ROOT / "labels" / split / f"{image_path.stem}.txt"
            for line in label_path.read_text(encoding="utf-8").splitlines():
                class_id, cx, cy, width, height = map(float, line.split())
                x1 = (cx - width / 2) * image.width
                y1 = (cy - height / 2) * image.height
                x2 = (cx + width / 2) * image.width
                y2 = (cy + height / 2) * image.height
                draw.rectangle((x1, y1, x2, y2), outline="#E63946", width=3)
                draw.text((x1 + 3, y1 + 3), CLASS_NAMES[int(class_id)], fill="#E63946")
            ax.imshow(image)
        ax.set_title(image_path.name, fontsize=9)
        ax.axis("off")
    for ax in axes.flat[len(sample):]:
        ax.axis("off")
    plt.suptitle(f"Converted {split} labels", fontsize=15)
    plt.tight_layout()
    plt.show()


show_yolo_samples("val")

## 6. Fine-tune a pretrained detector

Transfer learning gives the model a useful visual representation before it sees RPC's product classes. The official validation images are used during training for checkpoint selection and early stopping, but never for gradient updates. The test split remains untouched.

The image size, training duration, seed, and automatic memory-aware batch size are explicit. Ultralytics' per-image console chatter is disabled; the notebook prints a compact results table after training and saves full curves/checkpoints under the run directory. Start with this baseline, then change one factor at a time in a documented ablation.

In [ ]:
model = YOLO(MODEL_WEIGHTS)
train_started = time.time()
train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=-1,
    patience=10,
    optimizer="auto",
    cos_lr=True,
    close_mosaic=10,
    seed=SEED,
    deterministic=True,
    workers=2,
    amp=True,
    cache=False,
    plots=True,
    val=True,
    project=str(RUN_PROJECT),
    name="rpc_yolo26n_baseline",
    exist_ok=True,
    verbose=False,
)
print(f"Training wall time: {(time.time() - train_started) / 60:.1f} minutes")
TRAIN_RESULTS_CSV = RUN_PROJECT / "rpc_yolo26n_baseline" / "results.csv"
if TRAIN_RESULTS_CSV.is_file():
    epoch_history = pd.read_csv(TRAIN_RESULTS_CSV)
    display(epoch_history.tail(5).style.format(precision=4))
    print("Full epoch history:", TRAIN_RESULTS_CSV)

### Load the selected checkpoint

Use **best.pt**, chosen from validation performance, rather than the final epoch weights. Keeping an explicit path also makes it clear which checkpoint the prediction and price-verification code uses.

In [ ]:
RUN_DIR = RUN_PROJECT / "rpc_yolo26n_baseline"
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f"Best checkpoint not found: {BEST_WEIGHTS}. Inspect the training output above.")

best_model = YOLO(str(BEST_WEIGHTS))
print("Selected checkpoint:", BEST_WEIGHTS)

## 7. Validate localization and detection quality

Object detection needs more than a single accuracy number. We report precision, recall, mAP at IoU 0.50, and COCO-style mAP averaged over IoU 0.50–0.95. Per-class mAP helps reveal rare or confusable SKUs hidden by a global mean.

With the `quick` profile, these metrics are computed on a 1,000-image stratified validation sample, so treat them as an initial estimate. The default `full` profile evaluates all 6,000 official validation images.

These are detector metrics; checkout also cares about whether every item is counted correctly. The next sections handle that separately.

In [ ]:
val_detection_metrics = best_model.val(
    data=str(DATA_YAML_PATH), split="val", imgsz=IMAGE_SIZE,
    conf=PREDICTION_CONFIDENCE_FLOOR, iou=NMS_IOU, plots=True, verbose=False,
)

box_metrics = val_detection_metrics.box
print(f"Precision:      {box_metrics.mp:.4f}")
print(f"Recall:         {box_metrics.mr:.4f}")
print(f"mAP@0.50:       {box_metrics.map50:.4f}")
print(f"mAP@0.50:0.95:  {box_metrics.map:.4f}")

per_class_map = pd.DataFrame({"class": CLASS_NAMES, "mAP@0.50:0.95": box_metrics.maps})
display(per_class_map.sort_values("mAP@0.50:0.95").head(20).style.format({"mAP@0.50:0.95": "{:.3f}"}))

## 8. Select the checkout confidence on validation only

The model's confidence score is a ranking signal, not automatically a calibrated probability. Since a false extra product and a missed product both change the bill, choose one operating threshold on the checkout validation split using exact per-image class counts. Freeze it before opening the test split.

We scan a short threshold grid and report checkout exact-count accuracy plus mean absolute item-count error per image. This is an interpretable project metric; it should not be labeled as the official RPC cAcc/ACD unless evaluated with the authors' **rpctool** and its exact protocol.

In [ ]:
def yolo_image_counts(model, split: str, confidence_floor: float = PREDICTION_CONFIDENCE_FLOOR) -> tuple[dict, dict]:
    conversion = CONVERSION[split]
    image_name_to_id = conversion["output_name_to_id"]
    images_dir = YOLO_DATASET_ROOT / "images" / split
    detections_by_image = {int(image_id): [] for image_id in image_name_to_id.values()}
    results = model.predict(
        source=str(images_dir), imgsz=IMAGE_SIZE, conf=confidence_floor,
        iou=NMS_IOU, max_det=300, stream=True, verbose=False,
    )
    for result in results:
        output_name = Path(result.path).name
        image_id = image_name_to_id.get(output_name)
        if image_id is None:
            raise KeyError(f"Prediction image not found in conversion manifest: {output_name}")
        if result.boxes is None or len(result.boxes) == 0:
            continue
        class_ids = result.boxes.cls.detach().cpu().numpy().astype(int)
        confidence = result.boxes.conf.detach().cpu().numpy()
        detections_by_image[int(image_id)] = list(zip(class_ids.tolist(), confidence.tolist()))
    return detections_by_image, image_name_to_id


def ground_truth_count_vectors(split: str, image_ids: list[int]) -> np.ndarray:
    id_to_row = {image_id: index for index, image_id in enumerate(image_ids)}
    counts = np.zeros((len(image_ids), len(CLASS_NAMES)), dtype=np.int16)
    for ann in COCO[split]["annotations"]:
        image_id = int(ann["image_id"])
        if image_id not in id_to_row or ann.get("iscrowd", 0):
            continue
        class_index = COCO_ID_TO_CLASS[int(ann["category_id"])]
        counts[id_to_row[image_id], class_index] += 1
    return counts


def count_metrics_for_thresholds(detections: dict, image_ids: list[int], thresholds: list[float]) -> pd.DataFrame:
    truth = ground_truth_count_vectors("val", image_ids)
    rows = []
    for threshold in thresholds:
        prediction = np.zeros_like(truth)
        for row_index, image_id in enumerate(image_ids):
            for class_index, confidence in detections.get(image_id, []):
                if confidence >= threshold:
                    prediction[row_index, class_index] += 1
        absolute_error = np.abs(prediction - truth)
        rows.append({
            "confidence": threshold,
            "checkout exact-count accuracy": np.all(prediction == truth, axis=1).mean(),
            "mean absolute item-count error / image": absolute_error.sum(axis=1).mean(),
        })
    return pd.DataFrame(rows)


val_detections, val_name_to_id = yolo_image_counts(best_model, "val")
val_image_ids = sorted(val_name_to_id.values())
threshold_grid = [round(value, 2) for value in np.arange(0.05, 0.76, 0.05)]
threshold_results = count_metrics_for_thresholds(val_detections, val_image_ids, threshold_grid)
display(threshold_results.style.format({
    "confidence": "{:.2f}", "checkout exact-count accuracy": "{:.3%}",
    "mean absolute item-count error / image": "{:.3f}",
}))

best_row = threshold_results.sort_values(
    ["checkout exact-count accuracy", "mean absolute item-count error / image", "confidence"],
    ascending=[False, True, False],
).iloc[0]
CHECKOUT_CONFIDENCE = float(best_row["confidence"])
print(f"Frozen validation-selected confidence threshold: {CHECKOUT_CONFIDENCE:.2f}")

## 9. One final evaluation on the official test set (optional, resource intensive)

Run this section only after the checkpoint and confidence threshold are fixed. It converts the official 24,000-image checkout test split, evaluates detector mAP, and measures exact item-count performance using the frozen validation threshold. The test split must not be used to revise the model or threshold.

For a benchmark submission, also export predictions to COCO detection JSON and evaluate them with the RPC authors' **rpctool**; their official metrics include checkout accuracy and difficulty-specific scores.

In [ ]:
if PREPARE_OFFICIAL_TEST:
    test_conversion = convert_coco_split("test")
    CONVERSION["test"] = test_conversion
    test_metrics = best_model.val(
        data=str(DATA_YAML_PATH), split="test", imgsz=IMAGE_SIZE,
        conf=PREDICTION_CONFIDENCE_FLOOR, iou=NMS_IOU, plots=True, verbose=False,
    )
    print(f"Test mAP@0.50:       {test_metrics.box.map50:.4f}")
    print(f"Test mAP@0.50:0.95:  {test_metrics.box.map:.4f}")

    test_detections, test_name_to_id = yolo_image_counts(best_model, "test")
    test_image_ids = sorted(test_name_to_id.values())
    test_truth = ground_truth_count_vectors("test", test_image_ids)
    test_prediction = np.zeros_like(test_truth)
    for row_index, image_id in enumerate(test_image_ids):
        for class_index, confidence in test_detections.get(image_id, []):
            if confidence >= CHECKOUT_CONFIDENCE:
                test_prediction[row_index, class_index] += 1
    test_abs_error = np.abs(test_prediction - test_truth)
    print(f"Test checkout exact-count accuracy: {np.all(test_prediction == test_truth, axis=1).mean():.3%}")
    print(f"Test mean absolute item-count error / image: {test_abs_error.sum(axis=1).mean():.3f}")
else:
    print("Skipped the 24,000-image test split. Set PREPARE_OFFICIAL_TEST = True to run after model selection.")

## 10. Inspect a real checkout prediction

Review multiple easy, medium, and hard checkout examples. A visually plausible bounding box is not enough: verify the product identity and whether nearby or partially occluded items were missed or counted twice. Confidence threshold and NMS behavior should be reviewed on the validation split before test evaluation.

In [ ]:
val_image_dir = YOLO_DATASET_ROOT / "images" / "val"
sample_paths = sorted(path for path in val_image_dir.iterdir() if path.is_file())
sample_image = sample_paths[0]
sample_result = best_model.predict(
    source=str(sample_image), imgsz=IMAGE_SIZE, conf=CHECKOUT_CONFIDENCE,
    iou=NMS_IOU, max_det=300, verbose=False,
)[0]

annotated_rgb = sample_result.plot()[:, :, ::-1]
plt.figure(figsize=(13, 9))
plt.imshow(annotated_rgb)
plt.title(f"Checkout prediction | confidence ≥ {CHECKOUT_CONFIDENCE:.2f}")
plt.axis("off")
plt.show()

if sample_result.boxes is None or len(sample_result.boxes) == 0:
    print("No products passed the validation-selected confidence threshold.")
else:
    predicted_ids = sample_result.boxes.cls.detach().cpu().numpy().astype(int)
    predicted_counts = Counter(CLASS_NAMES[class_id] for class_id in predicted_ids)
    print("Detected product counts:", dict(predicted_counts))

## 11. Checkout summary with an explicit store catalog

The dataset contains product classes, not a trusted, current store price list. Prices therefore come from an external catalog supplied by the project owner. Add a mapping only after verifying that the model's exact class names correspond to those SKUs and that the price currency and effective date are correct.

Unmatched products are listed as **unpriced**. The output exposes a known-item subtotal and refuses to call it a complete bill when any item is unresolved. The example catalog is intentionally empty so no fictional prices are shown.

In [ ]:
# Example shape after loading a store-maintained catalog:
# SKU_PRICE_CATALOG = {"exact_model_class_name": {"currency": "EGP", "unit_price": 12.50}}
SKU_PRICE_CATALOG = {}


def checkout_summary(result, price_catalog: dict[str, dict], confidence: float) -> tuple[pd.DataFrame, dict]:
    item_rows = []
    if result.boxes is not None and len(result.boxes):
        class_ids = result.boxes.cls.detach().cpu().numpy().astype(int)
        scores = result.boxes.conf.detach().cpu().numpy()
        counts = Counter(CLASS_NAMES[class_id] for class_id, score in zip(class_ids, scores) if score >= confidence)
        for sku_name, quantity in sorted(counts.items()):
            price = price_catalog.get(sku_name)
            if price is None:
                item_rows.append({"product/SKU": sku_name, "quantity": quantity, "unit price": None,
                                  "currency": None, "subtotal": None, "status": "UNPRICED — catalog mapping required"})
            else:
                unit_price = float(price["unit_price"])
                currency = str(price["currency"])
                item_rows.append({"product/SKU": sku_name, "quantity": quantity, "unit price": unit_price,
                                  "currency": currency, "subtotal": quantity * unit_price, "status": "priced"})
    item_table = pd.DataFrame(item_rows, columns=["product/SKU", "quantity", "unit price", "currency", "subtotal", "status"])
    has_unpriced = not item_table.empty and (item_table["status"] != "priced").any()
    known_subtotal = item_table.loc[item_table["status"] == "priced", "subtotal"].sum() if not item_table.empty else 0.0
    currencies = sorted(item_table.loc[item_table["status"] == "priced", "currency"].dropna().unique()) if not item_table.empty else []
    complete = bool(not has_unpriced and len(currencies) <= 1 and not item_table.empty)
    summary = {
        "known_items_subtotal": float(known_subtotal),
        "currency": currencies[0] if len(currencies) == 1 else None,
        "bill_complete": complete,
        "status": "READY FOR HUMAN/STORE VALIDATION" if complete else "INCOMPLETE — unresolved SKU or currency",
    }
    return item_table, summary


checkout_items, checkout_totals = checkout_summary(sample_result, SKU_PRICE_CATALOG, CHECKOUT_CONFIDENCE)
display(checkout_items)
print(checkout_totals)

## 12. Interpretation, limitations, and next experiments

### What this baseline can establish

- Whether a fine-tuned detector can localize and classify RPC products in held-out cluttered checkout images.
- Which product classes are weak, and how the confidence threshold changes exact item counts.
- Whether a provided, verified SKU catalog can resolve detected classes to prices.

### What it cannot establish by itself

- A successful demo is not proof of production-grade checkout or trustworthy billing. The training/checkout domain gap, occlusion, packaging changes, camera angle, lighting, and previously unseen products all affect performance.
- Detection confidence is not a guaranteed probability of correctness. Calibration should be measured on representative store data before confidence is used for a financial decision.
- A superclass name is not a unique SKU. Use exact product identity plus a maintained catalog, and send low-confidence or unknown products to human review.
- The RPC dataset is restricted to non-commercial use under its published license. Obtain permission before commercial use.

### Research-backed follow-ups

1. **Augmentation ablation:** compare the baseline with paired-image composition / checkout-like synthesis, motivated by the 2024 checkout paper. Keep the same official validation set and report count accuracy by clutter level.
2. **Fine-grained product retrieval:** compare detector class scores with an image-embedding or OCR-assisted catalog retrieval branch for new packaging, inspired by the 2025 Visual RAG work.
3. **Confidence and review policy:** compare confidence thresholds on validation and report accuracy plus selective coverage (the share of items automatically accepted). Do not tune on test.
4. **Error analysis:** review confusion pairs, rare classes, missed small boxes, and duplicate detections. Keep a dated external validation set from the actual camera domain before deployment.

---

## References

1. Wei et al., **RPC: A Large-Scale and Fine-Grained Retail Product Checkout Dataset**, [project page and dataset protocol](https://rpc-dataset.github.io/).
2. **Solving automatic check-out with fine-tuned YOLO models**, Procedia Computer Science, 2024, [doi:10.1016/j.procs.2024.09.644](https://doi.org/10.1016/j.procs.2024.09.644).
3. Kuzucu et al., **On Calibration of Object Detectors: Pitfalls, Evaluation and Baselines**, ECCV 2024, [paper](https://arxiv.org/abs/2405.20459).
4. Lamm & Keuper, **A Visual RAG Pipeline for Few-Shot Fine-Grained Product Classification**, CVPR Workshops 2025, [paper](https://openaccess.thecvf.com/content/CVPR2025W/FGVC/html/Lamm_A_Visual_RAG_Pipeline_for_Few-Shot_Fine-Grained_Product_Classification_CVPRW_2025_paper.html).
5. [Ultralytics object-detection guide](https://docs.ultralytics.com/tasks/detect/) and [validation guide](https://docs.ultralytics.com/modes/val/).